In [17]:
from tradepy.data.loader import load_future
from tradepy.config.config import load_config, load_symbols, load_feature_config
from tradepy.data.cleaner import add_trading_date_by_gap
from tradepy.data.resampler import resample_ohlcv, daily_ohlcv_cummulative
from tradepy.features.generate import generate_features
from tradepy.features.quality_check import data_quality_report

In [18]:
symbols = load_symbols()

In [19]:
symbols

['AD',
 'BP',
 'CL',
 'EC',
 'ES',
 'GC',
 'MFXI',
 'NG',
 'NQ',
 'YM',
 'ZB',
 'ZN',
 'ZS']

In [20]:
ASSET = symbols[4]

In [21]:
cfg = load_config()

In [22]:
MINUTES             = cfg['sampling_minutes']        # 240
RETURN_HORIZON_MIN  = cfg['return_horizon_min']      # 2880

In [23]:
df, spec = load_future(ASSET)

In [24]:
df["datetime"] = df["datetime_utc"]

In [25]:
df = add_trading_date_by_gap(df)

In [26]:
df

,ticker,per,date,time,open,high,low,close,volume,openint,datetime,datetime_utc,session_id,trading_date
0,ES,I,1997-09-10,07:08:00,934.00,934.00,934.00,934.00,1,0,1997-09-10 13:08:00,1997-09-10 13:08:00,0,1997-09-10
1,ES,I,1997-09-10,07:15:00,933.75,933.75,933.75,933.75,1,0,1997-09-10 13:15:00,1997-09-10 13:15:00,0,1997-09-10
2,ES,I,1997-09-10,08:10:00,933.75,933.75,933.75,933.75,1,0,1997-09-10 14:10:00,1997-09-10 14:10:00,0,1997-09-10
3,ES,I,1997-09-10,08:13:00,934.00,934.00,934.00,934.00,1,0,1997-09-10 14:13:00,1997-09-10 14:13:00,0,1997-09-10
4,ES,I,1997-09-10,08:20:00,933.75,933.75,933.75,933.75,1,0,1997-09-10 14:20:00,1997-09-10 14:20:00,0,1997-09-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8631144,ES,I,2026-01-19,06:57:00,6917.00,6917.00,6916.75,6916.75,26,0,2026-01-19 12:57:00,2026-01-19 12:57:00,5073,2026-01-19
8631145,ES,I,2026-01-19,06:58:00,6917.00,6917.00,6916.75,6917.00,33,0,2026-01-19 12:58:00,2026-01-19 12:58:00,5073,2026-01-19
8631146,ES,I,2026-01-19,06:59:00,6917.00,6917.50,6916.75,6916.75,112,0,2026-01-19 12:59:00,2026-01-19 12:59:00,5073,2026-01-19
8631147,ES,I,2026-01-19,07:00:00,6916.75,6917.00,6916.25,6916.75,64,0,2026-01-19 13:00:00,2026-01-19 13:00:00,5073,2026-01-19


In [27]:
df_resampled = resample_ohlcv(df, period=f"{MINUTES}min")
df_cumulative = daily_ohlcv_cummulative(df_resampled)

In [28]:
config = load_feature_config()

In [29]:
config["global"]["sampling_minutes"] = MINUTES
config["global"]["return_horizon_min"] = RETURN_HORIZON_MIN
config["global"]["tick_size"] = spec["tick_size"]

In [30]:
df_final = generate_features(df_cumulative, config)

In [31]:
df_final

,datetime,open,high,low,close,volume,openint,ticker,per,trading_date,...,fib_dist_786,nearest_fib_level,dist_to_nearest_fib,swing_extension,bars_between_swing_highs,bars_between_swing_lows,swing_high_velocity,swing_low_velocity,bars_between_swings,swing_cycle_ratio
0,1997-09-10 16:00:00,934.00,934.25,933.75,934.25,11,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997-09-10 20:00:00,934.00,934.00,930.50,931.25,54,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997-09-11 00:00:00,931.25,932.75,925.00,928.00,1305,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1997-09-11 04:00:00,927.75,930.25,918.00,918.50,1550,0.0,ES,240min,1997-09-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1997-09-11 08:00:00,918.50,919.50,914.50,914.75,187,0.0,ES,240min,1997-09-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43757,2026-01-17 04:00:00,6986.75,6997.50,6976.75,6980.75,391672,0.0,ES,240min,2026-01-17,...,-0.599953,0.236,-0.049953,-0.813953,3.0,7.0,10.75,6.071429,3.0,0.333333
43758,2026-01-17 08:00:00,6980.75,6982.25,6973.50,6977.75,96050,0.0,ES,240min,2026-01-17,...,-0.415032,0.382,-0.011032,-0.629032,3.0,5.0,10.75,9.300000,2.0,1.000000
43759,2026-01-19 08:00:00,6918.25,6935.00,6911.00,6922.25,49191,0.0,ES,240min,2026-01-19,...,-1.608581,0.236,-1.058581,-1.822581,3.0,5.0,10.75,9.300000,2.0,1.500000
43760,2026-01-19 12:00:00,6922.50,6927.00,6914.25,6914.50,31659,0.0,ES,240min,2026-01-19,...,-1.775247,0.236,-1.225247,-1.989247,3.0,5.0,10.75,9.300000,2.0,2.000000


In [32]:
data_quality_report(df_final)

Data Quality Report
 - rows: 43762, cols: 188
 - top 20 NaN fractions:
swing_low              0.848224
swing_low_price        0.848224
swing_low_strength     0.848224
swing_high             0.846374
swing_high_price       0.846374
swing_high_strength    0.846374
ad_z                   0.026484
cmf                    0.026461
ad_sma_10              0.016613
ad_roc_10              0.003999
ad_roc_5               0.003839
ad_delta               0.003748
ad_dir                 0.003748
ad_rel                 0.002651
close_location         0.001897
body_pct               0.001897
wick_ratio             0.001897
volume_range           0.001897
obv_norm               0.001897
ad                     0.001897
dtype: float64

Suggestions:
 - Many columns have >50% NaNs; consider dropping or re-engineering them.
 - Drop or fix constant columns: ['openint', 'ticker', 'per', 'minute', 'minute_sin', 'minute_cos', 'is_post_break_hour', 'is_NY_open_hour']
 - `rolling_slope` may be slow; consider vect

{'rows': 43762,
 'cols': 188,
 'na_fraction': {'swing_low': 0.848224486997852,
  'swing_low_price': 0.848224486997852,
  'swing_low_strength': 0.848224486997852,
  'swing_high': 0.846373566107582,
  'swing_high_price': 0.846373566107582,
  'swing_high_strength': 0.846373566107582,
  'ad_z': 0.026484164343494355,
  'cmf': 0.026461313468305835,
  'ad_sma_10': 0.016612586262053836,
  'ad_roc_10': 0.003998903157990951,
  'ad_roc_5': 0.003838947031671313,
  'ad_delta': 0.003747543530917234,
  'ad_dir': 0.003747543530917234,
  'ad_rel': 0.0026507015218682874,
  'close_location': 0.0018966226406471368,
  'body_pct': 0.0018966226406471368,
  'wick_ratio': 0.0018966226406471368,
  'volume_range': 0.0018966226406471368,
  'obv_norm': 0.0018966226406471368,
  'ad': 0.0018966226406471368,
  'clv': 0.0018966226406471368,
  'ad_norm': 0.0018966226406471368,
  'upper_wick_pct': 0.0018966226406471368,
  'lower_wick_pct': 0.0018966226406471368,
  'pct_above_ma_50': 0.0011196928842374663,
  'obv_z': 0.0

In [33]:
df_final.columns.tolist()

['datetime',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'openint',
 'ticker',
 'per',
 'trading_date',
 'open_day',
 'high_cum',
 'low_cum',
 'volume_cum',
 'sma_20',
 'ema_20',
 'dema',
 'kama',
 'ema_fast',
 'ema_slow',
 'macd',
 'signal',
 'histogram',
 'adx',
 'rsi',
 'slowk',
 'slowd',
 'cmo',
 'williams_r',
 'cci',
 'roc_10',
 'roc_1',
 'roc_5',
 'roc_20',
 'bb_middle',
 'bb_std',
 'bb_upper',
 'bb_lower',
 'true_range',
 'range',
 'range_mean',
 'hv',
 'realized_vol_20',
 'obv',
 'obv_norm',
 'obv_roc_5',
 'obv_roc_10',
 'obv_ema_10',
 'obv_sma_10',
 'obv_z',
 'obv_dir',
 'obv_rel',
 'obv_delta',
 'vpt',
 'clv',
 'ad',
 'ad_norm',
 'ad_roc_5',
 'ad_roc_10',
 'ad_ema_10',
 'ad_sma_10',
 'ad_z',
 'ad_delta',
 'ad_dir',
 'ad_rel',
 'volume_range',
 'volume_price',
 'mfi',
 'cmf',
 'efi',
 'vwap',
 'avg_volume',
 'relative_volume',
 'volume_z',
 'volume_sum_20',
 'volume_delta',
 'pct_above_ma_20',
 'pct_above_ma_50',
 'slope_10',
 'tr_dir',
 'body',
 'wick_ratio',
 'is_doji',
